# VADAR Visual Agent – Quick Demo

This notebook walks you through:
1. Installing dependencies
2. Initialising the VADARAgent
3. Detecting objects and estimating depth in a sample image
4. Asking a spatial reasoning question
5. Visualising the results

**Before running:** make sure `OPENAI_API_KEY` is set in your environment or in a `.env` file.

## 0 · Install / verify dependencies

In [ ]:
# Uncomment the line below to install all requirements in one shot:
# !pip install -r requirements.txt

# Verify setup
import subprocess, sys
result = subprocess.run([sys.executable, 'verify_setup.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

## 1 · Load environment and initialise the agent

In [ ]:
import os
from pathlib import Path

# Load .env file if it exists
try:
    from dotenv import load_dotenv
    load_dotenv()
    print('✓ .env loaded')
except ImportError:
    print('⚠  python-dotenv not installed – set OPENAI_API_KEY manually')

api_key = os.getenv('OPENAI_API_KEY', '')
if api_key:
    print('✓ OPENAI_API_KEY is set')
else:
    print('⚠  OPENAI_API_KEY is not set – code-generation step will be skipped')

In [ ]:
from vadar_agent import VADARAgent

agent = VADARAgent(api_key)
print('✓ VADARAgent initialised')

## 2 · Load a sample image

In [ ]:
import urllib.request
from PIL import Image
import matplotlib.pyplot as plt

SAMPLE_URL = (
    'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/'
    'Cat03.jpg/320px-Cat03.jpg'
)
SAMPLE_PATH = Path('sample_images') / 'demo_sample.jpg'
SAMPLE_PATH.parent.mkdir(exist_ok=True)

if not SAMPLE_PATH.exists():
    urllib.request.urlretrieve(SAMPLE_URL, SAMPLE_PATH)
    print(f'✓ Downloaded sample image → {SAMPLE_PATH}')
else:
    print(f'✓ Using cached image at {SAMPLE_PATH}')

# Display
img = Image.open(SAMPLE_PATH)
plt.figure(figsize=(6, 4))
plt.imshow(img)
plt.title('Sample image')
plt.axis('off')
plt.show()

## 3 · Object detection & depth estimation

In [ ]:
print('Running object detection (may take a minute on first run while models download)…')
detections = agent.vision_models.detect_objects(img)
print(f'\nDetected {len(detections)} object(s):')
for d in detections:
    print(f"  • {d['label']:20s}  score={d['score']:.3f}  box={d['box']}")

In [ ]:
import numpy as np

print('Running depth estimation…')
depth_map = agent.vision_models.estimate_depth(img)
print(f'Depth map shape : {depth_map.shape}')
print(f'Min / max depth : {depth_map.min():.3f} / {depth_map.max():.3f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(img)
axes[0].set_title('Original image')
axes[0].axis('off')
axes[1].imshow(depth_map, cmap='viridis')
axes[1].set_title('Depth map (0 = close, 1 = far)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 4 · Full scene analysis with spatial objects

In [ ]:
import matplotlib.patches as patches

scene = agent.analyze_image(str(SAMPLE_PATH))
print(f'Scene contains {len(scene.objects)} spatial object(s):\n')
for obj in scene.objects:
    print(
        f'  {obj.label:20s}  depth={obj.depth_value:.3f}'
        f'  center={obj.center}  conf={obj.confidence:.3f}'
    )

# Visualise with bounding boxes
img_arr = np.array(img)
h, w = img_arr.shape[:2]

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(img_arr)
for obj in scene.objects:
    x0, y0, x1, y1 = obj.bbox
    rect = patches.Rectangle(
        (x0 * w, y0 * h), (x1 - x0) * w, (y1 - y0) * h,
        linewidth=2, edgecolor='red', facecolor='none'
    )
    ax.add_patch(rect)
    ax.text(
        x0 * w, max(y0 * h - 5, 0),
        f"{obj.label}\ndepth={obj.depth_value:.2f}",
        color='red', fontsize=9,
        bbox=dict(facecolor='yellow', alpha=0.7)
    )
ax.set_title('Detected objects with depth information')
ax.axis('off')
plt.tight_layout()
plt.show()

## 5 · Spatial reasoning question (requires OPENAI_API_KEY)

In [ ]:
if not api_key:
    print('⚠  Skipping – OPENAI_API_KEY not set.')
else:
    question = 'Which detected object appears to be the closest to the camera?'
    print(f'Question: {question}\n')

    result = agent.answer_question(question, str(SAMPLE_PATH))

    print(f'Answer : {result["answer"]}')
    print(f'Status : {result["status"]}')
    print('\nGenerated code:')
    print('─' * 50)
    print(result['code'])
    print('─' * 50)

## 6 · (Optional) Run the evaluation benchmark

In [ ]:
if not api_key:
    print('⚠  Skipping benchmark – OPENAI_API_KEY not set.')
else:
    from evaluate_benchmark import BenchmarkEvaluator

    evaluator = BenchmarkEvaluator(api_key, output_dir='benchmark_results_notebook')

    test_cases = [
        {
            'sample_id': 'demo_001',
            'image_path': str(SAMPLE_PATH),
            'questions': [
                'Which object is furthest from the camera?',
                'Are there multiple objects in the scene?',
            ],
        }
    ]

    evaluator.run_evaluation(test_cases)
    summary = evaluator.generate_summary_report()
    print('\nBenchmark complete.  Results saved to benchmark_results_notebook/')

---
🎉 **That's it!**  You can now load your own images and ask any spatial reasoning question.